In [1]:
from dotenv import load_dotenv
import os
load_dotenv()
api_key = os.getenv('GroqAPIKey')

### **STEP-1** INSTALL THE PACKAGES - We already have the 'Groq' package insatlled.

### **STEP-2** IMPORT THE PACKAGES

In [30]:
from groq import Groq
from openai import OpenAI
import json

In [3]:
#client = Groq(api_key=api_key)
client = OpenAI(api_key=api_key, base_url="https://api.groq.com/openai/v1")

In [4]:
message = [
    {
        "role":"system",
        "content":"You are an useful agent. I want you to use tools when needed."
    },
    {
        "role":"user",
        "content":"How much is 345 divided by 5"
    }
]


response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=message
)

print(response)
print(f"Answer is::: {response.choices[0].message.content}")

ChatCompletion(id='chatcmpl-d2540f28-b061-4925-87b6-57f8fe76d44c', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='345 divided by\u202f5 equals **69**.', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning='The user asks "How much is 345 divided by 5". Simple division: 345 / 5 = 69. So answer: 69.'))], created=1783443022, model='openai/gpt-oss-120b', object='chat.completion', service_tier='on_demand', system_fingerprint='fp_4140daa9c2', usage=CompletionUsage(completion_tokens=52, prompt_tokens=98, total_tokens=150, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=33, rejected_prediction_tokens=None), prompt_tokens_details=None, queue_time=0.522673024, prompt_time=0.003886884, completion_time=0.108221621, total_time=0.112108505), usage_breakdown=None, x_groq={'id': 'req_01kwyqsqp6ez58h3bdv9ftfdx7', 'seed': 194

In [5]:
# This will fail, because, we have given a system promt to agent saying to use tools when needed, but actually, we haven't provided any tools. So, for my quest, it actually need a tool (whether api). Since, we haven't provided tools, it will not be able to connect. So it returned with msg: "Tool choice is none, but model called a tool".
#Here, a point to be noted is for messageUser, it didn't fail, because here we have removed the system prompt. We just used the LLM from user perspective, and didn't give any persona.
messageSystem = [
    {
        "role":"system",
        "content":"You are an useful agent. I want you to use tools when needed."
    },
    {
        "role":"user",
        "content":"What is the temperature in Paris no? "
    }
]

messageUser = [
    
    {
        "role":"user",
        "content":"What is the temperature in Paris no? "
    }
]


responseUser = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=messageUser
)

print(f"responseUser:: {responseUser}")
print(f"Answer is::: {responseUser.choices[0].message.content}")

responseSystem = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=messageSystem
)


print(f"responseSystem:: {responseSystem}")
print(f"Answer is::: {responseSystem.choices[0].message.content}")

responseUser:: ChatCompletion(id='chatcmpl-3c67c2d8-0a91-4ef5-8ced-5a70d18a2f72', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='I’m not able to pull live weather data, so I can’t give you the current temperature in Paris right this moment.  \n\nFor an up‑to‑date reading, you can check any of these reliable sources:\n\n* **Weather websites** –\u202f[Weather.com](https://weather.com), [AccuWeather](https://www.accuweather.com), or the French service [Météo France](https://www.meteofrance.com).\n* **Mobile apps** –\u202fthe default weather app on iOS/Android, or dedicated apps like\u202fWeather Underground, WeatherBug, etc.\n* **Voice assistants** –\u202fask Siri, Google Assistant, or Alexa “What’s the temperature in Paris right now?”\n* **Search engines** –\u202ftype “Paris weather” into Google or Bing and the current temperature is displayed at the top of the results.\n\nIf you need a forecast (hourly or 7‑day) or historical tempera

BadRequestError: Error code: 400 - {'error': {'message': 'Tool choice is none, but model called a tool', 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '{"name": "web.run", "arguments": {\n  "cursor": 0,\n  "id": "https://wttr.in/Paris?format=%t"\n}}'}}

### Creating Functions
***Syntax*** : def function_name(parameter: Type) -> ReturnType:

In [7]:
def calculator(expression:str) -> str:
    #eval will evaluate the expression.
    result = eval(expression)
    return str(result)


res = calculator("45*67+23")
print(f"45*67+23 :: {res}")
print(f"type of res: {type(res)}")
    

45*67+23 :: 3038
type of res: <class 'str'>


In [66]:
def mockWeatherData(city:str) -> str:
    cityUpper = city.upper()
    whetherData ={
        "CHENNAI":"39c, drizzling",
        "MUMBAI":"34c, Polluted and dirty",
        "DELHI":"29c, Hevealy polluted, bad AQI",
        "KERALA":"22c, Plesent, calm and fresh whether"
    }

    if cityUpper in whetherData:
        return str(whetherData.get(cityUpper))
    
# city='delhi'
# positiveTemp = mockWhetherData(city)
# print(f"Whether in {city}: {positiveTemp}")

# city='bangalore'
# negativeTemp = mockWhetherData(city)
# print(f"Whether in {city}: {negativeTemp}")


    

### The above are the tools that we were talking about. These are the hands that we give to our agents to perform tasks.

### HOW TIO FEED THESE TOOLS TO OUR AGENTS?
We need to create a json explaining our function as a tool to the agent.

In [67]:
tools=[
    {
        "type":"function",
        "function":{
            "name":"calculator", #func name
            "description":"A function to evaluate a mathematical expression. use this for math computions.", #"Returns recipe of a dish."
            "parameters":{
                "type":"object",
                "properties":{
                    "expression":{
                    "type":"string",
                    "description":"The expression to evaluate. Eg: '(45+78)*167'"
                    }
                },
                "required":["expression"]               
            }
        }
    },
    {
        "type":"function",
        "function":{
            "name":"mockWeatherData", #func name
            "description":"A function to which returns the weather details of the city. Use this for weather related data.",
            "parameters":{
                "type":"object",
                "properties":{
                    "city":{
                    "type":"string",
                    "description":"This is the cuty for which weather details has to be returned. Eg: 'Chennai'"
                    }
                },
                "required":["city"]               
            }
        }
    }
]

In [10]:
print(f"Tools defined for our agent: {tools}")

Tools defined for our agent: [{'type': 'function', 'function': {'name': 'calculator', 'description': 'A function to evaluate a mathematical expression. use this for math computions.', 'parameters': {'type': 'object', 'properties': {'expression': {'type': 'string', 'description': "The expression to evaluate. Eg: '(45+78)*167'"}}, 'required': ['expression']}}}, {'type': 'function', 'function': {'name': 'mockWhetherData', 'description': 'A function to which returns the whether details of the city. Use this for whether related data.', 'parameters': {'type': 'object', 'properties': {'city': {'type': 'string', 'description': "This is the cuty for which whether details has to be returned. Eg: 'Chennai'"}}, 'required': ['city']}}}]


### Calling LLM with tools defined.

In [46]:
messageExp = [
    {
        "role":"system",
        "content":"You are an useful agent. I want you to use tools when needed."
    },
    {
        "role":"user",
        "content":"How much is 345 divided by 5"
    }
]

messageWhether = [
    {
        "role":"system",
        "content":"You are an useful agent. I want you to use tools when needed."
    },
    {
        "role":"user",
        "content":"delhi"
    }
]

respwithTools = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=messageExp,
    tools=tools
)

#If you see below, our finish_reason=""tool_calls, meaning we have a tool that does our work. So, the agent didn't give any result.
print(f"respwithTools choice:: {respwithTools.choices[0]}")
print(f"respwithTools choice:: {respwithTools.choices[0].message.content}")

''' Here, even b4, w/o using tools, we received ans. But with tools, it is acting differently. Why???
Modern reasoning models are trained to prefer tools when an appropriate tool exists.

Think of the model's decision process as:
User asks question -> Do I have a suitable tool? -> YES -> Use the tool -> NO -> Answer directly.

Our tolls matching with the user query heavily depends on the Function name, Description (most important), Parameter names, Parameter descriptions, The user's query, The conversation history. We need to give proper description, otherwise, the model will not pick it up. '''

respwithTools choice:: Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='fc_f5f6cf88-512e-42eb-92f4-bf5eda6f0d98', function=Function(arguments='{"expression":"345/5"}', name='calculator'), type='function')], reasoning='User asks: "How much is 345 divided by 5". Simple division. Compute 345/5 = 69. Use calculator tool.'))
respwithTools choice:: None


" Here, even b4, w/o using tools, we received ans. But with tools, it is acting differently. Why???\nModern reasoning models are trained to prefer tools when an appropriate tool exists.\n\nThink of the model's decision process as:\nUser asks question -> Do I have a suitable tool? -> YES -> Use the tool -> NO -> Answer directly.\n\nOur tolls matching with the user query heavily depends on the Function name, Description (most important), Parameter names, Parameter descriptions, The user's query, The conversation history. We need to give proper description, otherwise, the model will not pick it up. "

In [ ]:
mockWeatherData('delhi')

'29c, Hevealy polluted, bad AQI'

In [12]:
print(f"Finish Reason: {respwithTools.choices[0].finish_reason}")
print(f"Tool calls: {respwithTools.choices[0].message.tool_calls}")

print("-------LLM DECISION---------")
toolCalls=respwithTools.choices[0].message.tool_calls

for i in range(0,len(toolCalls)):
    tool = toolCalls[i]
    print(f"Tool name: {tool.function.name}")
    print(f"Arguments: {tool.function.arguments}")
    print(f"tool id: {tool.id}")


Finish Reason: tool_calls
Tool calls: [ChatCompletionMessageFunctionToolCall(id='fc_110f4e48-646d-4e28-b7c2-b08077a9860b', function=Function(arguments='{"expression":"345/5"}', name='calculator'), type='function')]
-------LLM DECISION---------
Tool name: calculator
Arguments: {"expression":"345/5"}
tool id: fc_110f4e48-646d-4e28-b7c2-b08077a9860b


In [69]:
availableTools=   {
        "calculator":calculator,
        "mockWeatherData":mockWeatherData
    }

arguments = json.loads(tool.function.arguments)
print(arguments)

{'expression': '345/5'}


In [15]:
result = availableTools[tool.function.name](**arguments)
print(result)

69.0


### Feeding the tool response to LLM.

In [16]:
messageExp.append(respwithTools.choices[0].message)
messageExp.append({
    "role":"tool",
    "tool_call_id":tool.id,
    "content":result
})

finalrespTools = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=messageExp,
    tools=tools
)

#print(f"finalrespTools:: {finalrespTools}")
print(f"finalrespTools:: {finalrespTools.choices[0].message.content}")

finalrespTools:: 345 divided by 5 equals **69**.


### Putting th entire code together - The below block is an Agent.

In [70]:
availableTools=   {
        "calculator":calculator,
        "mockWeatherData":mockWeatherData
    }

messageExp = [
    {
        "role":"system",
        "content":"You are an useful agent. I want you to use tools when needed."
    },
    {
        "role":"user",
        "content":"How much is 345 divided by 5"
    }
]


respwithTools = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=messageExp,
    tools=tools
)

print(f"Finish Reason: {respwithTools.choices[0].finish_reason}")
print(f"Tool calls: {respwithTools.choices[0].message.tool_calls}")

print("-------LLM DECISION---------")
toolCalls=respwithTools.choices[0].message.tool_calls

for i in range(0,len(toolCalls)):
    tool = toolCalls[i]
    print(f"Tool name: {tool.function.name}")
    print(f"Arguments: {tool.function.arguments}")
    print(f"tool id: {tool.id}")

arguments = json.loads(tool.function.arguments)
print(arguments)

result = availableTools[tool.function.name](**arguments)
print(result)

messageExp.append(respwithTools.choices[0].message)
messageExp.append({
    "role":"tool",
    "tool_call_id":tool.id,
    "content":result
})

finalrespTools = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=messageExp,
    tools=tools
)

#print(f"finalrespTools:: {finalrespTools}")
print(f"finalrespTools:: {finalrespTools.choices[0].message.content}")



Finish Reason: tool_calls
Tool calls: [ChatCompletionMessageFunctionToolCall(id='fc_85b10b5c-860c-4d90-9579-cc10f5a015e2', function=Function(arguments='{"expression":"345/5"}', name='calculator'), type='function')]
-------LLM DECISION---------
Tool name: calculator
Arguments: {"expression":"345/5"}
tool id: fc_85b10b5c-860c-4d90-9579-cc10f5a015e2
{'expression': '345/5'}
69.0
finalrespTools:: 345 divided by 5 equals **69**.


All the response from tools or functions has to be str, because all the LLM's are trained on text data. So, we need to convert the response from our tools to str before feeding it to LLM.

### THE AGENT LOOP - CORE OF AGENTS WORKFLOW
It is a loop that keeps running until the agent is able to solve the problem. The agent will keep calling the LLM and the tools until it is able to solve the problem. The agent will keep track of the number of iterations and will stop if it reaches the maximum number of iterations.

In [35]:
def autonomousAgent(userMsg,maxIterations=5,verbose=True):
    ''' 
    Here, we are trying to run an autonomous agent that accepts user message, maxIteration(Hard stops - max tool calls b4 stopping),
    Verbose - it is a system specvific parameter, it is like setting our logger to debug mode.
    '''
    message=[
        {
            "role":"system",
            "content":"Your are an useful agent. Use tools when needed."            
        },
        {
            "role":"user",
            "content":userMsg

        }
    ]

    if verbose:
        print("="*80)
        print(f"User Message: {userMsg}")
        print("="*80)

    for step in range(0,maxIterations):
        #STEP 1- Ask the LLM.
        response = client.chat.completions.create(
            model="openai/gpt-oss-120b",
            messages=message,
            tools=tools
        )

        choice = response.choices[0]  
    
        # CASE 1: LLM is done - it has the final answer.
        if choice.finish_reason == 'stop':
            if verbose:
                print(f"The LLM is ready with the final answer:: {choice.message.content}")
                return choice.message.content
        if choice.message.tool_calls:
            #Add step-1's message, to the "message", sending LLM the conversation history.
            message.append(choice.message)
    
            #Executing the LLM's returned tool, for fetching O/P for user query
            for toolCall in choice.message.tool_calls:
                funcName = toolCall.function.name
                toolId = toolCall.id
                args = json.loads(toolCall.function.arguments)
    
                if verbose:
                    print(f"Step - {step + 1}: Calling {funcName} with arguments {args}")
    
                    if funcName in availableTools:
                        toolResult = availableTools[funcName](**args)
                    else:
                        toolResult = f"Unkown tool -{funcName}"
    
                    if verbose:
                        print(f"Tool Result: {toolResult}")
    
                    #Appending Tool's result to LLM's message.
                    message.append({
                        "role":"tool",
                        "tool_call_id":toolId,
                        "content":toolResult
                    })      
    return "Max iterations reached, without fetching result."

In [47]:
userMsg = "What is the result of (56*98)+56-90"
responseFromAgent = autonomousAgent(userMsg=userMsg)

User Message: What is the result of (56*98)+56-90
Step - 1: Calling calculator with arguments {'expression': '(56*98)+56-90'}
Tool Result: 5454
The LLM is ready with the final answer:: The result of \((56 \times 98) + 56 - 90\) is **5,454**.


In [ ]:
mockWeatherData('Delhi')

'29c, Hevealy polluted, bad AQI'

In [71]:
userMsg = "What is the weather in delhi?"
responseFromAgent = autonomousAgent(userMsg=userMsg)

User Message: What is the weather in delhi?
Step - 1: Calling mockWeatherData with arguments {'city': 'Delhi'}
Tool Result: 29c, Hevealy polluted, bad AQI
The LLM is ready with the final answer:: The current weather in Delhi is **29 °C**, with heavy pollution and a poor air quality index (AQI). Take precautions if you’re planning to be outdoors.
